### `Text Classification`

In [1]:
import pandas as pd
pd.set_option('display.max_colwidth', 100000)
from IPython.display import display
import os

# sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [2]:
# a helper for displaying the DataFrame
def highlight_col(x, df):
    # set by condition
    pos_mask = (df['label'] == 'pos')
    neg_mask = (df['label'] == 'neg')
    x = pd.DataFrame('', index=df.index, columns=df.columns)
    x.loc[pos_mask] = 'background-color: #e6ffe6'
    x.loc[neg_mask] = 'background-color: #ffe6e6'
    return x    

* `Load the data` [Here](https://www.kaggle.com/datasets/mksaad/arabic-sentiment-twitter-corpus)

In [4]:
# read tsv files for train
TRAIN_POS_PATH = os.path.join(os.getcwd(), 'data', 'train_Arabic_tweets_positive_20190413.tsv')
TRAIN_NEG_PATH = os.path.join(os.getcwd(), 'data', 'train_Arabic_tweets_negative_20190413.tsv')

df_train_pos = pd.read_csv(TRAIN_POS_PATH, sep='\t', header=None)
df_train_neg = pd.read_csv(TRAIN_NEG_PATH, sep='\t', header=None)

# Concate both
df_train = pd.concat([df_train_pos, df_train_neg], ignore_index=True)
df_train.columns = ['label', 'tweet']

df_train.head()

,label,tweet
0,pos,نحن الذين يتحول كل ما نود أن نقوله إلى دعاء لله، لا تبحثوا فينا عن قوة، إننا مكسورون، القوة التي…
1,pos,وفي النهاية لن يبقىٰ معك آحدإلا من رأىٰ الجمال في روحك أماالمنبهرون بالمظا…
2,pos,من الخير نفسه 💛
3,pos,#زلزل_الملعب_نصرنا_بيلعب كن عالي الهمه ولا ترضى بغير القمه مجرد ساعات لاستعادة الصداره💛💙 الوصول إلى القمه مهارة ت…
4,pos,الشيء الوحيد الذي وصلوا فيه للعالمية هو : المسيار ..! . ترى كانوا يشجعون ريال مدريد ضد النصر 🤣


In [5]:
# No need more for them after concatenating
del df_train_pos, df_train_neg

In [6]:
# See Highlited DF with my custom function
df_tmp = df_train.sample(5)
df_tmp.style.apply(lambda x: highlight_col(x, df_tmp), axis=None)

,label,tweet
21288,pos,ده انتي مش هتحتويهم بس ده انتي هترضعيهم كمان .. قلب الام 😇
37071,neg,فيه كل الروايح الي اكرهها شوي عود وشوي زي ريحة الدخان 😷
888,pos,🎤 من حفلة الرياض 🎤 تركي الميزاني محمد العازمي بخيت السناني محمد العلوني
242,pos,ماتمت مناقشته أمس 😏 #FFM
4145,pos,انهض ياعميد من قلب هلالي 💙 #الاتحاد


In [7]:
# read tsv files for train
TEST_POS_PATH = os.path.join(os.getcwd(), 'data', 'test_Arabic_tweets_positive_20190413.tsv')
TEST_NEG_PATH = os.path.join(os.getcwd(), 'data', 'test_Arabic_tweets_negative_20190413.tsv')

df_test_pos = pd.read_csv(TEST_POS_PATH, sep='\t', header=None)
df_test_neg = pd.read_csv(TEST_NEG_PATH, sep='\t', header=None)

# Concate both
df_test = pd.concat([df_test_pos, df_test_neg], ignore_index=True)
df_test.columns = ['label', 'tweet']

df_test.head()

,label,tweet
0,pos,#الهلال_الاهلي فوز هلالي مهم الحمد لله 💙 زوران كان بيسلم المباراة بعد تبديل كارييو بإنتظار الإتحاد بكرة يارب يار…
1,pos,صباحك خيرات ومسرات 🌸
2,pos,"#تأمل قال الله ﷻ :- _*​﴿بواد غير ذي زرع ﴾*_ 💫💫 ✍ "" ~ومع ذلك هتف بالدعاء ﴿وارزقهم من الثمرات ﴾ مهماكانت ظرو…"
3,pos,😂😂 يا جدعان الرجاله اللي فوق ال دول خطر ع تويتر وربنا 😂مش اسلوب كل يومين يدخلي واحد قد جدي علشان يشقطني 😒 😹و عند…
4,pos,رساله صباحيه : 💛 اللهم اسألك التوفيق في جميع امورنا واكتب لنا الفردوس نحن ووالدينا وجميع موتى المسلمين برحمتك يا ارحم الراحمين


In [8]:
# No need more for them after concatenating
del df_test_pos, df_test_neg

In [9]:
# See Highlited DF with my custom function
df_tmp = df_test.sample(5)
df_tmp.style.apply(lambda x: highlight_col(x, df_tmp), axis=None)

,label,tweet
9381,neg,وكيل الشيطان #قطر لماذا #الدوحه و في عهد #تنظيم_الحمدين دعمها للمعاض السعودي الاماراتي الموريتاني الليبي الصومالي ا…
415,pos,بلاش كسوف بقى حضرتك 😅 صباح الخير صباح الفل والياسمين انا قمر ولا على 🤔😅
55,pos,وياك يارب يا اخي 👍
9594,neg,على الاقل هذي خطوة جباره منهم افضل من المتلحفين بالبرانص ورا الكيبورد، وبالعين ب…
1886,pos,اللهم ارزقنا أجمل مما تمنينا وأكثر مما توقعنا وأفضل مما دعونا 💙


----

* `Baseline model (using pipeline)`

In [10]:
# Vectorizing and then model
vect = CountVectorizer(max_features=15000, encoding='utf-8') # arabic () 
clf = LogisticRegression(max_iter=10000)

# combine them to a pipeline
pipe = Pipeline(steps=[
        ('vectorizer', vect),
        ('classifier', clf)
    ])

# fitting to train data
pipe.fit(df_train['tweet'], df_train['label'])

Pipeline(steps=[('vectorizer', CountVectorizer(max_features=15000)),
                ('classifier', LogisticRegression(max_iter=10000))])

In [11]:
df_train['label'].value_counts()   # balanced dataset

label
pos    22761
neg    22514
Name: count, dtype: int64

* `Test the Baseline`

In [12]:
# predict on test
y_pred_test = pipe.predict(df_test['tweet'])
print(f'Accuracy is {accuracy_score(df_test["label"], y_pred_test) * 100:.3f} %')

Accuracy is 77.127 %


* `Try Tfidf with some Processing`

In [13]:
# Using TFIDF and SVC in one pipeline
# char_wb: will generate character n-grams within each word separately and will not span across the boundaries of different word
vect = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=0.01, max_df=0.5, max_features=10000)
clf = LinearSVC()
pipe_tfidf = Pipeline(steps=[
        ('vectorizer', vect),
        ('classifier', clf)
    ])
pipe_tfidf.fit(df_train['tweet'], df_train['label'])

Pipeline(steps=[('vectorizer',
                 TfidfVectorizer(analyzer='char_wb', max_df=0.5,
                                 max_features=10000, min_df=0.01,
                                 ngram_range=(3, 5))),
                ('classifier', LinearSVC())])

In [14]:
# predict on test
y_pred_test = pipe_tfidf.predict(df_test['tweet'])
print(f'Accuracy is {accuracy_score(df_test["label"], y_pred_test) * 100:.3f} %')

Accuracy is 83.811 %


In [15]:
import joblib
joblib.dump(pipe_tfidf, 'model.pkl')

['model.pkl']

---